## Họ và tên: Huỳnh Quảng Tín
## MSSV: 24110353
Link github: https://github.com/AIVIETNAM-AIO-tinbmt79/Tri_tue_nhan_tao/blob/master/README.md 


In [7]:
%pip install pygame

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import sys
import random
from collections import deque
import heapq

#### LOGIC ####

class Node:
    def __init__(self, state, parent, action, cost, name):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost
        self.name = name

def percept():

    def is_solvable(matrix):
        arr = []
        for row in matrix:
            for val in row:
                if val != 0:
                    arr.append(val)

        inversions = 0
        for i in range(len(arr)):
            for j in range(i + 1, len(arr)):

                if arr[i] > arr[j]:
                    inversions += 1
        return inversions % 2 == 0

    while True:

        list_num = list(range(9))

        random.shuffle(list_num)

        matrix = [
            list_num[i:i+3]
            for i in range(0, 9, 3)
        ]

        if is_solvable(matrix):
            return matrix

def interpret_input(matrix):
    for i in range(3):
        for j in range(3):
            if matrix[i][j] == 0:
                return (i, j)

def rules(state):
    x, y = state
    actions = []
    if x > 0: actions.append("UP")
    if x < 2: actions.append("DOWN")
    if y > 0: actions.append("LEFT")
    if y < 2: actions.append("RIGHT")
    return actions

def check_now(matrix):
    return 1 if matrix == [[1, 2, 3], [4, 5, 6], [7, 8, 0]] else 0

def action(chosen, matrix, state):
    x, y = state
    new_matrix = [row[:] for row in matrix]
    value1 = new_matrix[x][y]

    if chosen == "UP":
        new_matrix[x][y] = new_matrix[x-1][y]
        new_matrix[x-1][y] = value1
        return new_matrix, (x-1, y)
    elif chosen == "DOWN":
        new_matrix[x][y] = new_matrix[x+1][y]
        new_matrix[x+1][y] = value1
        return new_matrix, (x+1, y)
    elif chosen == "RIGHT":
        new_matrix[x][y] = new_matrix[x][y+1]
        new_matrix[x][y+1] = value1
        return new_matrix, (x, y+1)
    else:  # LEFT
        new_matrix[x][y] = new_matrix[x][y-1]
        new_matrix[x][y-1] = value1
        return new_matrix, (x, y-1)

def get_solution_path(goal_node_name, all_nodes):
    path = []
    current_name = goal_node_name
    while current_name is not None:
        node = all_nodes[current_name]
        path.append((node.state, node.action))
        current_name = node.parent
    path.reverse()
    return path

# BFS1: Kiem tra goal khi lay node ra khoi hang cho
def solve_8_puzzle_bfs1(initial_matrix):
    all_nodes = {}
    root_name = "A"
    root = Node(state=initial_matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root
    queue = deque([root])
    reached = set()
    in_queue = set([str(root.state)])
    limit = 181440
    step = 0
    child_id = 1
    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)

    while len(queue) > 0 and step < limit:
        current_node = queue.popleft()
        in_queue.remove(str(current_node.state))
        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        else:
            reached.add(str(current_node.state))

        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            if new_matrix_str not in reached and new_matrix_str not in in_queue:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, current_node.cost + 1, child_name)
                queue.append(child_node)
                in_queue.add(new_matrix_str)
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

# BFS2: Kiem tra goal ngay khi tao child
def solve_8_puzzle_bfs2(initial_matrix):
    all_nodes = {}
    root_name = "A"
    root = Node(state=initial_matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root

    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)

    queue = deque([root])
    reached = set()
    in_queue = set([str(root.state)])
    limit = 181440
    step = 0
    child_id = 1

    while len(queue) > 0 and step < limit:
        current_node = queue.popleft()
        in_queue.remove(str(current_node.state))
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        else:
            reached.add(str(current_node.state))

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            if new_matrix_str not in reached and new_matrix_str not in in_queue:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, current_node.cost + 1, child_name)

                if check_now(new_matrix) == 1:
                    all_nodes[child_name] = child_node
                    return get_solution_path(child_name, all_nodes)
                queue.append(child_node)
                in_queue.add(new_matrix_str)
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

# DFS2: Kiem tra goal ngay khi tao child
def solve_8_puzzle_dfs2(initial_matrix):
    all_nodes = {}
    root_name = "A"
    root = Node(state=initial_matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root

    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)

    stack = deque([root])
    reached = set()
    in_stack = set([str(root.state)])
    limit = 181440
    step = 0
    child_id = 1

    while len(stack) > 0 and step < limit:
        current_node = stack.pop()
        in_stack.remove(str(current_node.state))
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        else:
            reached.add(str(current_node.state))

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            if new_matrix_str not in reached and new_matrix_str not in in_stack:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, current_node.cost + 1, child_name)

                if check_now(new_matrix) == 1:
                    all_nodes[child_name] = child_node
                    return get_solution_path(child_name, all_nodes)
                stack.append(child_node)
                in_stack.add(new_matrix_str)
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

# IDS: kiem tra goal ngay khi tao child
def solve_8_puzzle_ids(initial_matrix):
    all_nodes = {}
    child_id = [1]  

    def depth_limited_search(root_matrix, limit_depth):
        all_nodes.clear()
        child_id[0] = 1
        root_name = "A"
        root = Node(state=root_matrix, parent=None, action=None, cost=0, name=root_name)
        all_nodes[root_name] = root

        # Kiem tra goal tai trang thai goc
        if check_now(root_matrix) == 1:
            return get_solution_path(root_name, all_nodes), False  
        
        stack = deque([root])
        in_stack = set([str(root.state)])
        reached = set()
        result_cutoff = False

        while len(stack) > 0:
            current_node = stack.pop()
            in_stack.discard(str(current_node.state))

            if check_now(current_node.state) == 1:
                return get_solution_path(current_node.name, all_nodes), False

            reached.add(str(current_node.state))

            if current_node.cost >= limit_depth:
                result_cutoff = True
                continue

            zero_pos = interpret_input(current_node.state)
            valid_actions = rules(zero_pos)

            for act in valid_actions:
                new_matrix, _ = action(act, current_node.state, zero_pos)
                new_matrix_str = str(new_matrix)

                if new_matrix_str not in reached and new_matrix_str not in in_stack:
                    child_name = f"Node_{child_id[0]}"
                    child_node = Node(new_matrix, current_node.name, act, current_node.cost + 1, child_name)

                    # Kiem tra goal ngay khi tao child 
                    if check_now(new_matrix) == 1:
                        all_nodes[child_name] = child_node
                        return get_solution_path(child_name, all_nodes), False

                    stack.append(child_node)
                    in_stack.add(new_matrix_str)
                    all_nodes[child_name] = child_node
                    child_id[0] += 1

        return None, result_cutoff

    for depth in range(0, 181440):
        result, is_cutoff = depth_limited_search(initial_matrix, depth)
        if result is not None:
            return result
        if not is_cutoff:
            return None  # Khong co loi giai (khong bi cat, het node)

    return None

def solve_8_puzzle_ucs(initial_matrix):
    def calculate_cost(matrix):
        finish_matrix = [[1,2,3],[4,5,6],[7,8,0]]
        cost = 0
        for i in range(3):
            for j in range(3):
                if matrix[i][j] != 0 and matrix[i][j] != finish_matrix[i][j]:
                    cost += 1 
        return cost
    
    all_nodes = {}
    root_name = "A"
    
    cost_root = calculate_cost(initial_matrix) 
    root = Node(state=initial_matrix, parent=None, action=None, cost=cost_root, name=root_name)
    all_nodes[root_name] = root
    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)
    
    queue = []
    heapq.heappush(queue, (root.cost, 0, root))
    
    reached = {}
    limit = 181440
    step = 0
    child_id = 1
    
    while len(queue) > 0 and step < limit:
        _, _, current_node = heapq.heappop(queue)
        state_str = str(current_node.state)

        if state_str in reached and reached[state_str] <= current_node.cost:
            continue

        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        
        reached[state_str] = current_node.cost  # lưu cost tốt nhất
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)
            new_cost = current_node.cost + calculate_cost(new_matrix)

            if new_matrix_str not in reached or reached[new_matrix_str] > new_cost:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, new_cost, child_name)
                heapq.heappush(queue, (child_node.cost, child_id, child_node))
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

def solve_8_puzzle_greedy(initial_matrix):
    #Số ô sai
    def calculate_cost(matrix):
        finish_matrix = [[1,2,3],[4,5,6],[7,8,0]]
        cost = 0
        for i in range(3):
            for j in range(3):
                if matrix[i][j] != 0 and matrix[i][j] != finish_matrix[i][j]:
                    cost += 1 
        return cost
    
    all_nodes = {}
    root_name = "A"
    
    cost_root = calculate_cost(initial_matrix) 
    root = Node(state=initial_matrix, parent=None, action=None, cost=cost_root, name=root_name)
    all_nodes[root_name] = root
    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)
    
    queue = []
    heapq.heappush(queue, (root.cost, 0, root))
    
    reached = set()
    limit = 181440
    step = 0
    child_id = 1
    
    while len(queue) > 0 and step < limit:
        _, _, current_node = heapq.heappop(queue)
        state_str = str(current_node.state)

        # Bỏ qua nếu đã xử lý với cost tốt hơn
        if state_str in reached:
            continue
        reached.add(state_str)

        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)
            cost = calculate_cost(new_matrix)

            if new_matrix_str not in reached:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, cost, child_name)
                heapq.heappush(queue, (child_node.cost, child_id, child_node))
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

def solve_8_puzzle_A_star(initial_matrix):
    def calculate_cost(matrix):
        cost = 0
        for i in range(3):
            for j in range(3):
                val = matrix[i][j]
                if val != 0:
                    target_i = (val - 1) // 3
                    target_j = (val - 1) % 3
                    cost += abs(i - target_i) + abs(j - target_j)
        return cost

    all_nodes = {}
    root_name = "A"

    h_root = calculate_cost(initial_matrix)
    g_root = 0
    root = Node(state=initial_matrix, parent=None, action=None, cost=g_root + h_root, name=root_name)
    root.g = g_root
    all_nodes[root_name] = root

    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)

    queue = []
    heapq.heappush(queue, (root.cost, 0, root))

    reached = {}       
    limit = 181440
    child_id = 1

    while len(queue) > 0 and len(reached) < limit:

        _, _, current_node = heapq.heappop(queue)

        state_str = str(current_node.state)
        #gặp trùng nếu cost nó lớn hơn = thì continue
        if state_str in reached and reached[state_str] <= current_node.g:
            continue

        reached[state_str] = current_node.g  

        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)

        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            g = current_node.g + 1
            if new_matrix_str not in reached or reached[new_matrix_str] > g:
                h = calculate_cost(new_matrix)
                f = g + h
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, f, child_name)
                child_node.g = g
                heapq.heappush(queue, (child_node.cost, child_id, child_node))
                all_nodes[child_name] = child_node
                child_id += 1

    return None

def solve_8_puzzle_IDA_star(initial_matrix):
    def calculate_cost(matrix):
        cost = 0
        for i in range(3):
            for j in range(3):
                val = matrix[i][j]
                if val != 0:
                    target_i = (val - 1) // 3
                    target_j = (val - 1) % 3
                    cost += abs(i - target_i) + abs(j - target_j)
        return cost

    def search_f(current_node, f_limit):
        f = current_node.g + calculate_cost(current_node.state)
        # Nếu f vượt quá giới hạn, dừng nhánh này và trả về f để cập nhật limit mới
        if f > f_limit:
            return f, None
        
        if check_now(current_node.state) == 1:
            return "FOUND", current_node.name

        min_val = float('inf')
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        nonlocal child_id
        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            g = current_node.g + 1
            if new_matrix_str in reached and reached[new_matrix_str] <= g:
                continue

            reached[new_matrix_str] = g

            child_name = f"Node_{child_id}"
            child_node = Node(new_matrix, current_node.name, act, 0, child_name)
            child_node.g = g
            all_nodes[child_name] = child_node
            child_id += 1
            
            t, result_node_name = search_f(child_node, f_limit)
            
            if t == "FOUND":
                return "FOUND", result_node_name
            
            if t < min_val:
                min_val = t
            
            if new_matrix_str in reached:
                del reached[new_matrix_str]
                
        return min_val, None

    all_nodes = {}
    root_name = "A"
    
    h_root = calculate_cost(initial_matrix)
    root = Node(state=initial_matrix, parent=None, action=None, cost=h_root, name=root_name)
    root.g = 0
    all_nodes[root_name] = root
    reached = {str(initial_matrix): 0}
    
    f_limit = h_root
    child_id = 1
    
    while f_limit != float('inf'):
        t, target_name = search_f(root, f_limit)
        
        if t == "FOUND":
            return get_solution_path(target_name, all_nodes)

        if t == float('inf'):
            return None

        reached = {str(initial_matrix): 0}
        f_limit = t

    return None

def solve_8_puzzle_SHC(initial_matrix):
    def calculate_cost(matrix):
        cost = 0
        for i in range(3):
            for j in range(3):
                val = matrix[i][j]
                if val != 0:
                    target_i = (val - 1) // 3
                    target_j = (val - 1) % 3
                    cost += abs(i - target_i) + abs(j - target_j)
        return cost

    all_nodes = {}
    root_name = "A"

    cost_root = calculate_cost(initial_matrix)
    root = Node(state=initial_matrix, parent=None, action=None, cost=cost_root, name=root_name)
    all_nodes[root_name] = root

    current_node = root
    child_id = 1

    while True:
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        found_better = False

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            next_cost = calculate_cost(new_matrix)

            if next_cost < current_node.cost:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, next_cost, child_name)
                all_nodes[child_name] = child_node
                child_id += 1

                current_node = child_node
                found_better = True
                break
        if not found_better:
            if check_now(current_node.state) == 1:
                return get_solution_path(current_node.name, all_nodes)
            return None  

def solve_8_puzzle_BHC(initial_matrix):
    def calculate_cost(matrix):
        cost = 0
        for i in range(3):
            for j in range(3):
                val = matrix[i][j]
                if val != 0:
                    target_i = (val - 1) // 3
                    target_j = (val - 1) % 3
                    cost += abs(i - target_i) + abs(j - target_j)
        return cost
    all_nodes = {}
    root_name = "A"

    cost_root = calculate_cost(initial_matrix)
    root = Node(state=initial_matrix, parent=None, action=None, cost=cost_root, name=root_name)
    all_nodes[root_name] = root

    current_node = root
    child_id = 1

    while True:
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        best_matrix = None
        best_act = None
        best_cost = current_node.cost
        found_better = False
        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            next_cost = calculate_cost(new_matrix)
            if next_cost < best_cost:
                best_cost = next_cost
                best_matrix = new_matrix
                best_act = act
                found_better = True

        if not found_better:
            if check_now(current_node.state) == 1:
                return get_solution_path(current_node.name, all_nodes)
            return None  

        child_name = f"Node_{child_id}"
        child_node = Node(best_matrix, current_node.name, best_act, best_cost, child_name)
        all_nodes[child_name] = child_node
        child_id += 1
        current_node = child_node

In [9]:
import pygame
import sys

WHITE       = (245, 245, 245)
BLACK       = (30,  30,  30)
DARK_BG     = (14,  16,  26)
TILE_CLR    = (64, 120, 210)
TILE_EMPTY  = (40,  44,  66)
GRAY        = (90,  95, 120)
ACCENT      = (255, 190,  60)

LOG_BG      = (18,  20,  34)
LOG_BORDER  = (50,  55,  85)
LOG_HEAD    = (40,  60, 120)
LOG_EVEN    = (24,  27,  44)
LOG_ODD     = (28,  32,  50)
LOG_TEXT    = (190, 205, 255)
LOG_ACT     = (255, 190,  60)

UNINFORMED_ALGOS = [
    {"key": "BFS1",   "label": "BFS 1",  "desc": "Breadth-First Search 1",   "color": (243, 156, 18)},
    {"key": "BFS2",   "label": "BFS 2",  "desc": "Breadth-First Search 2",   "color": (52,  152, 219)},
    {"key": "DFS2",   "label": "DFS 2",  "desc": "Depth-First Search",       "color": (46,  204, 113)},
    {"key": "IDS",    "label": "IDS",    "desc": "Iterative Deepening",      "color": (155,  89, 182)},
    {"key": "UCS",    "label": "UCS",    "desc": "Uniform-Cost Search",      "color": (230, 126,  34)},
]

INFORMED_ALGOS = [
    {"key": "GREEDY", "label": "Greedy", "desc": "Greedy Best-First Search", "color": (231,  76,  60)},
    {"key": "A*",     "label": "A*",     "desc": "A* Search",                "color": (240, 116,  90)},
    {"key": "IDA*",   "label": "IDA*",   "desc": "IDA* Search",              "color": (200, 100,  40)},
]

CLIMBING = [
    {"key": "SHC", "label": "SHC", "desc": "Simple Hill Climbing", "color": (204, 129, 210)},
    {"key": "BHC", "label": "BHC", "desc": "Best Hill Climbing",   "color": (232, 119, 230)},
]

ALL_ALGOS = UNINFORMED_ALGOS + INFORMED_ALGOS + CLIMBING

ACTION_LABELS = {
    "UP":    "Di chuyen len",
    "DOWN":  "Di chuyen xuong",
    "LEFT":  "Di chuyen trai",
    "RIGHT": "Di chuyen phai",
    None:    "Trang thai ban dau",
}

def load_font(size, bold=False):
    candidates = ["segoeui", "arialuni", "notosans", "dejavusans", "freesans", "liberation sans", "tahoma"]
    for name in candidates:
        try:
            f = pygame.font.SysFont(name, size, bold=bold)
            if f.render("Test", True, (255,255,255)).get_width() > 10:
                return f
        except:
            pass
    return pygame.font.SysFont("arial", size, bold=bold)

def draw_board(screen, matrix, font, ox, oy, tile=88, margin=8):
    for i in range(3):
        for j in range(3):
            val = matrix[i][j]
            rx  = ox + j * (tile + margin)
            ry  = oy + i * (tile + margin)
            rect = pygame.Rect(rx, ry, tile, tile)
            if val == 0:
                pygame.draw.rect(screen, TILE_EMPTY, rect, border_radius=14)
                pygame.draw.rect(screen, (60, 65, 95), rect, 2, border_radius=14)
            else:
                shadow = pygame.Rect(rx + 3, ry + 5, tile, tile)
                pygame.draw.rect(screen, (20, 45, 110), shadow, border_radius=14)
                pygame.draw.rect(screen, TILE_CLR, rect, border_radius=14)
                hi = pygame.Rect(rx + 6, ry + 5, tile - 12, 10)
                pygame.draw.rect(screen, (100, 160, 255), hi, border_radius=6)
                txt = font.render(str(val), True, WHITE)
                screen.blit(txt, txt.get_rect(center=rect.center))

def draw_button(screen, rect, label, font, style="default",
                hovered=False, pressed=False, disabled=False):
    offset = 0
    if disabled:
        bg, border, txt_c = (35, 38, 58), (55, 60, 90), GRAY
    elif style == "run":
        if pressed:      bg, border, txt_c = (30,160,80),   (60,200,120),  WHITE;         offset=2
        elif hovered:    bg, border, txt_c = (60,220,130),  (80,255,160),  (10,40,20)
        else:            bg, border, txt_c = (146,219,65),  (100,255,100), (10,40,20)
    elif style == "warning":
        if pressed:      bg, border, txt_c = (211,84,0),    (250,170,40),  WHITE;          offset=2
        elif hovered:    bg, border, txt_c = (250,170,40),  (255,200,80),  WHITE
        else:            bg, border, txt_c = (243,156,18),  (255,180,50),  WHITE
    elif style == "danger":
        if pressed:      bg, border, txt_c = (192,57,43),   (240,90,75),   WHITE;          offset=2
        elif hovered:    bg, border, txt_c = (240,90,75),   (250,110,90),  WHITE
        else:            bg, border, txt_c = (231,76,60),   (250,100,80),  WHITE
    else:
        bg, border, txt_c = TILE_CLR, (100,200,255), WHITE

    pygame.draw.rect(screen, bg,     rect, border_radius=9)
    pygame.draw.rect(screen, border, rect, width=2, border_radius=9)
    txt      = font.render(label, True, txt_c)
    txt_rect = txt.get_rect(center=rect.center)
    txt_rect.y += offset
    screen.blit(txt, txt_rect)

def draw_log_panel(screen, log_entries, scroll_offset, panel_rect, font_head, font_row):
    px, py, pw, ph = panel_rect
    pygame.draw.rect(screen, LOG_BG,     panel_rect, border_radius=12)
    pygame.draw.rect(screen, LOG_BORDER, panel_rect, width=1, border_radius=12)

    header_h = 34
    hdr_rect  = pygame.Rect(px, py, pw, header_h)
    pygame.draw.rect(screen, LOG_HEAD, hdr_rect, border_radius=12)
    pygame.draw.rect(screen, LOG_HEAD, pygame.Rect(px, py + header_h//2, pw, header_h//2))
    hdr_txt = font_head.render("Log cac buoc thuc hien", True, WHITE)
    screen.blit(hdr_txt, hdr_txt.get_rect(center=hdr_rect.center))

    row_h        = 28
    content_area = pygame.Rect(px, py + header_h, pw, ph - header_h)
    clip         = screen.get_clip()
    screen.set_clip(content_area)

    for idx, entry in enumerate(log_entries):
        row_y = py + header_h + idx * row_h - scroll_offset
        if row_y + row_h < py + header_h or row_y > py + ph:
            continue
        row_rect = pygame.Rect(px + 2, row_y, pw - 4, row_h - 2)
        bg = (28, 55, 38) if idx == len(log_entries)-1 else (LOG_EVEN if idx%2==0 else LOG_ODD)
        pygame.draw.rect(screen, bg, row_rect, border_radius=4)

        screen.blit(font_row.render(f"B{entry['step']:03d}", True, ACCENT),      (px+8,   row_y+6))
        screen.blit(font_row.render(entry['action_label'],   True, LOG_ACT),     (px+56,  row_y+6))
        rows_str = " | ".join(" ".join(str(v) for v in r) for r in entry['matrix'])
        screen.blit(font_row.render(rows_str,                True, LOG_TEXT),    (px+200, row_y+6))

    screen.set_clip(clip)

    total_h = len(log_entries) * row_h
    view_h  = ph - header_h
    if total_h > view_h:
        ratio = view_h / total_h
        bar_h = max(24, int(view_h * ratio))
        bar_y = py + header_h + int(scroll_offset / max(1, total_h) * view_h)
        pygame.draw.rect(screen, (70,80,120),
                         pygame.Rect(px + pw - 9, bar_y, 6, bar_h), border_radius=3)

def build_layout(W, H):
    """Tính toán lại toàn bộ layout khi cửa sổ thay đổi kích thước."""
    BOARD_X, BOARD_Y = 30, 100
    BOARD_SIZE = 3 * 88 + 2 * 8          # 280

    CTRL_X   = BOARD_X + BOARD_SIZE + 30  # 340
    CTRL_Y   = BOARD_Y
    CTRL_W   = 380

    ALGO_COLS   = 3
    ALGO_BTN_W  = (CTRL_W - 8 * (ALGO_COLS - 1)) // ALGO_COLS   # ~120
    ALGO_BTN_H  = 35
    ALGO_PAD    = 8

    algo_rects = {}
    current_y  = CTRL_Y

    # --- nhóm 1 ---
    uninfo_lbl_y = current_y - 22
    for i, algo in enumerate(UNINFORMED_ALGOS):
        ax = CTRL_X + (i % ALGO_COLS) * (ALGO_BTN_W + ALGO_PAD)
        ay = current_y + (i // ALGO_COLS) * (ALGO_BTN_H + ALGO_PAD)
        algo_rects[algo["key"]] = pygame.Rect(ax, ay, ALGO_BTN_W, ALGO_BTN_H)
    current_y += ((len(UNINFORMED_ALGOS) + ALGO_COLS - 1) // ALGO_COLS) * (ALGO_BTN_H + ALGO_PAD) + 8

    # --- nhóm 2 ---
    info_lbl_y = current_y
    current_y += 22
    for i, algo in enumerate(INFORMED_ALGOS):
        ax = CTRL_X + (i % ALGO_COLS) * (ALGO_BTN_W + ALGO_PAD)
        ay = current_y + (i // ALGO_COLS) * (ALGO_BTN_H + ALGO_PAD)
        algo_rects[algo["key"]] = pygame.Rect(ax, ay, ALGO_BTN_W, ALGO_BTN_H)
    current_y += ((len(INFORMED_ALGOS) + ALGO_COLS - 1) // ALGO_COLS) * (ALGO_BTN_H + ALGO_PAD) + 8

    # --- nhóm 3 ---
    climbing_lbl_y = current_y
    current_y += 22
    for i, algo in enumerate(CLIMBING):
        ax = CTRL_X + (i % ALGO_COLS) * (ALGO_BTN_W + ALGO_PAD)
        ay = current_y + (i // ALGO_COLS) * (ALGO_BTN_H + ALGO_PAD)
        algo_rects[algo["key"]] = pygame.Rect(ax, ay, ALGO_BTN_W, ALGO_BTN_H)
    current_y += ((len(CLIMBING) + ALGO_COLS - 1) // ALGO_COLS) * (ALGO_BTN_H + ALGO_PAD) + 6

    desc_y    = current_y;  current_y += 20

    # --- nút hành động ngang ---
    BTN_Y   = current_y + 6
    BTN_H   = 44
    BTN_GAP = 8
    BTN_W   = (CTRL_W - BTN_GAP * 2) // 3

    btn_run   = pygame.Rect(CTRL_X,                         BTN_Y, BTN_W, BTN_H)
    btn_new   = pygame.Rect(CTRL_X + BTN_W + BTN_GAP,       BTN_Y, BTN_W, BTN_H)
    btn_reset = pygame.Rect(CTRL_X + (BTN_W + BTN_GAP) * 2, BTN_Y, BTN_W, BTN_H)

    status_y  = BTN_Y + BTN_H + 10

    log_rect  = pygame.Rect(
        CTRL_X + CTRL_W + 20,
        20,
        W - (CTRL_X + CTRL_W + 20) - 20,
        H - 40
    )

    return {
        "BOARD_X": BOARD_X, "BOARD_Y": BOARD_Y,
        "CTRL_X":  CTRL_X,  "CTRL_W":  CTRL_W,
        "algo_rects":     algo_rects,
        "uninfo_lbl_y":   uninfo_lbl_y,
        "info_lbl_y":     info_lbl_y,
        "climbing_lbl_y": climbing_lbl_y,
        "desc_y":         desc_y,
        "btn_run":   btn_run,
        "btn_new":   btn_new,
        "btn_reset": btn_reset,
        "status_y":  status_y,
        "log_rect":  log_rect,
    }

def main_gui():
    pygame.init()
    W, H = 1150, 640
    screen = pygame.display.set_mode((W, H), pygame.RESIZABLE)
    pygame.display.set_caption("8-Puzzle Solver")

    font_tile  = load_font(44, bold=True)
    font_big   = load_font(20, bold=True)
    font_med   = load_font(16, bold=True)
    font_small = load_font(14)
    font_log_h = load_font(14, bold=True)
    font_log_r = load_font(12)
    font_desc  = load_font(12)
    clock = pygame.time.Clock()

    try:    initial_matrix = percept()
    except: initial_matrix = [[1,2,3],[4,5,6],[7,0,8]]

    current_matrix = [row[:] for row in initial_matrix]
    selected_algo  = "BFS1"
    status_msg     = "San sang! Chon thuat toan va nhan RUN."
    status_type    = "info"
    is_fullscreen  = False

    is_animating = False
    solution_path, anim_index, last_update = [], 0, 0
    DELAY = 650
    log_entries, log_scroll = [], 0

    layout = build_layout(W, H)

    running = True
    while running:
        screen.fill(DARK_BG)
        mouse         = pygame.mouse.get_pos()
        mouse_pressed = pygame.mouse.get_pressed()[0]

        # --- sử dụng layout hiện tại ---
        algo_rects     = layout["algo_rects"]
        BOARD_X        = layout["BOARD_X"]
        BOARD_Y        = layout["BOARD_Y"]
        CTRL_X         = layout["CTRL_X"]
        CTRL_W         = layout["CTRL_W"]
        btn_run        = layout["btn_run"]
        btn_new        = layout["btn_new"]
        btn_reset      = layout["btn_reset"]
        status_y       = layout["status_y"]
        LOG_RECT       = layout["log_rect"]
        uninfo_lbl_y   = layout["uninfo_lbl_y"]
        info_lbl_y     = layout["info_lbl_y"]
        climbing_lbl_y = layout["climbing_lbl_y"]
        desc_y         = layout["desc_y"]

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

            if event.type == pygame.VIDEORESIZE:
                W, H   = event.w, event.h
                screen = pygame.display.set_mode((W, H), pygame.RESIZABLE)
                layout = build_layout(W, H)

            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_F11:
                    is_fullscreen = not is_fullscreen
                    if is_fullscreen:
                        screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN)
                        W, H   = screen.get_size()
                    else:
                        W, H   = 1150, 640
                        screen = pygame.display.set_mode((W, H), pygame.RESIZABLE)
                    layout = build_layout(W, H)

            if event.type == pygame.MOUSEWHEEL:
                log_scroll = max(0, log_scroll - event.y * 20)

            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1 and not is_animating:
                for algo in ALL_ALGOS:
                    if algo_rects[algo["key"]].collidepoint(event.pos):
                        selected_algo = algo["key"]

                if btn_new.collidepoint(event.pos):
                    try:    initial_matrix = percept()
                    except: pass
                    current_matrix = [row[:] for row in initial_matrix]
                    solution_path, log_entries, log_scroll = [], [], 0
                    status_msg, status_type = "Trang thai moi! Nhan RUN de giai.", "info"

                elif btn_reset.collidepoint(event.pos):
                    current_matrix = [row[:] for row in initial_matrix]
                    solution_path, log_entries, log_scroll = [], [], 0
                    status_msg, status_type = "Da quay lai trang thai ban dau.", "info"

                elif btn_run.collidepoint(event.pos):
                    current_matrix = [row[:] for row in initial_matrix]
                    log_entries, log_scroll = [], 0
                    status_msg, status_type = f"Dang tinh toan ({selected_algo})...", "info"
                    result = None
                    try:
                        if   selected_algo == "BFS1":   result = solve_8_puzzle_bfs1(current_matrix)
                        elif selected_algo == "BFS2":   result = solve_8_puzzle_bfs2(current_matrix)
                        elif selected_algo == "DFS2":   result = solve_8_puzzle_dfs2(current_matrix)
                        elif selected_algo == "IDS":    result = solve_8_puzzle_ids(current_matrix)
                        elif selected_algo == "UCS":    result = solve_8_puzzle_ucs(current_matrix)
                        elif selected_algo == "GREEDY": result = solve_8_puzzle_greedy(current_matrix)
                        elif selected_algo == "A*":     result = solve_8_puzzle_A_star(current_matrix)
                        elif selected_algo == "IDA*":   result = solve_8_puzzle_IDA_star(current_matrix)
                        elif selected_algo == "SHC":    result = solve_8_puzzle_SHC(current_matrix)
                        elif selected_algo == "BHC":    result = solve_8_puzzle_BHC(current_matrix)
                    except NameError:
                        pass

                    if result:
                        solution_path  = result
                        is_animating   = True
                        anim_index     = 0
                        last_update    = pygame.time.get_ticks()
                        status_msg     = f"Tim thay! {len(solution_path)-1} buoc ({selected_algo})"
                        status_type    = "success"
                    else:
                        status_msg, status_type = "Khong tim thay duong di!", "error"

        # --- animation ---
        if is_animating:
            now = pygame.time.get_ticks()
            if now - last_update > DELAY:
                mat, act        = solution_path[anim_index]
                current_matrix  = mat
                label           = ACTION_LABELS.get(act, act if act else "?")
                log_entries.append({"step": anim_index, "action_label": label, "matrix": mat})
                total_h = len(log_entries) * 28
                view_h  = LOG_RECT[3] - 34
                if total_h > view_h:
                    log_scroll = total_h - view_h
                anim_index += 1
                last_update = now
                if anim_index >= len(solution_path):
                    is_animating = False
                    status_msg, status_type = "HOAN THANH!", "success"

        # ===================== VẼ =====================

        # Board
        lbl = font_small.render("Trang thai hien tai", True, (120,135,185))
        screen.blit(lbl, (BOARD_X, BOARD_Y - 25))
        draw_board(screen, current_matrix, font_tile, BOARD_X, BOARD_Y)

        # Label nhóm
        screen.blit(font_small.render("Tim kiem khong co thong tin:", True, (120,135,185)),
                    (CTRL_X, uninfo_lbl_y))
        screen.blit(font_small.render("Tim kiem co thong tin:", True, (120,135,185)),
                    (CTRL_X, info_lbl_y))
        screen.blit(font_small.render("Leo doi:", True, (120,135,185)),
                    (CTRL_X, climbing_lbl_y))

        # Nút algo
        for algo in ALL_ALGOS:
            rect       = algo_rects[algo["key"]]
            base_color = algo["color"]
            is_sel     = algo["key"] == selected_algo
            is_hov     = rect.collidepoint(mouse) and not is_animating
            is_prs     = is_hov and mouse_pressed
            offset     = 0

            if is_sel:
                bg, border, txt_c = (20,24,38), base_color, base_color
            elif is_prs:
                bg = tuple(max(0,c-40) for c in base_color)
                border, txt_c, offset = bg, WHITE, 2
            elif is_hov:
                bg = tuple(min(255,c+30) for c in base_color)
                border, txt_c = bg, WHITE
            else:
                bg, border, txt_c = base_color, base_color, WHITE

            pygame.draw.rect(screen, bg,     rect, border_radius=6)
            pygame.draw.rect(screen, border, rect, width=2, border_radius=6)
            lbl_surf = font_med.render(algo["label"], True, txt_c)
            lbl_r    = lbl_surf.get_rect(center=rect.center)
            lbl_r.y += offset
            screen.blit(lbl_surf, lbl_r)

        # Mô tả algo đang chọn
        for algo in ALL_ALGOS:
            if algo["key"] == selected_algo:
                screen.blit(font_desc.render(algo["desc"], True, (130,150,200)),
                            (CTRL_X, desc_y))
                break

        # Nút hành động
        draw_button(screen, btn_run,   "RUN",           font_big,   style="run",
                    hovered=btn_run.collidepoint(mouse)   and not is_animating,
                    pressed=btn_run.collidepoint(mouse)   and mouse_pressed and not is_animating,
                    disabled=is_animating)
        draw_button(screen, btn_new,   "Trang thai moi", font_small, style="warning",
                    hovered=btn_new.collidepoint(mouse)   and not is_animating,
                    pressed=btn_new.collidepoint(mouse)   and mouse_pressed and not is_animating,
                    disabled=is_animating)
        draw_button(screen, btn_reset, "Quay lai",       font_small, style="danger",
                    hovered=btn_reset.collidepoint(mouse) and not is_animating,
                    pressed=btn_reset.collidepoint(mouse) and mouse_pressed and not is_animating,
                    disabled=is_animating)

        # Status
        s_color = (46,204,113) if status_type=="success" else (220,70,70) if status_type=="error" else (160,175,220)
        st_bg   = pygame.Rect(CTRL_X-4, status_y-4, CTRL_W+8, 26)
        pygame.draw.rect(screen, (24,28,48), st_bg, border_radius=6)
        screen.blit(font_small.render(status_msg, True, s_color), (CTRL_X, status_y))

        if is_animating:
            tick = (pygame.time.get_ticks() // 400) % 4
            screen.blit(font_small.render(f"Dang chay{'.'*(tick+1)}", True, ACCENT),
                        (CTRL_X, status_y + 24))

        # Hint F11
        screen.blit(font_desc.render("F11: Phong to / Thu nho", True, (60,70,100)),
                    (BOARD_X, H - 22))

        draw_log_panel(screen, log_entries, log_scroll, LOG_RECT, font_log_h, font_log_r)

        pygame.display.flip()
        clock.tick(60)

    pygame.quit()
    sys.exit()

if __name__ == "__main__":
    main_gui()

SystemExit: 